# 05 – Factor scoring & backtest

**Question:** would ranking S&P 500 members by these signals have picked better stocks?

**Method** – every month end since 2011:
1. members of the index *at that date* (point in time, see notebook 03)
2. factor values using only data public at that date (10-K filing date, month-end price)
3. 5 equal-weighted portfolios by score (Q1 worst … Q5 best), held for one month

**How to read the results**
* a working factor: returns rise **monotonically** Q1 → Q5, long-short (Q5 − Q1) clearly positive
* **t-stat** > ~2 = unlikely to be luck (with 8 factors tested, expect one false positive at t ≈ 2 by chance alone)
* **IC** (rank correlation score vs. next-month return) of 0.02–0.05 is already useful in practice

⚠️ **Biases that remain:** missing delisted companies (see coverage in notebook 03, ~72 % in 2011 → ~97 % in 2025),
no transaction costs unless stated, financials identified by *today's* sub-industry, data-mining risk.

In [ ]:
from IPython.display import display
import pandas as pd

from stockpred.config import load_settings
from stockpred.factors import build_factor_table, composite_score, FACTOR_DIRECTION
from stockpred.backtest import quintile_backtest, perf_stats, cumulative
from stockpred.plots import backtest_figure, quintile_bar_figure, ic_figure

settings = load_settings()
P = settings.processed_dir
membership = pd.read_parquet(P / "membership.parquet")
panel = pd.read_parquet(P / "fundamentals_full.parquet")
dcf = pd.read_parquet(P / "dcf_values_full.parquet")
tri = pd.read_parquet(P / "prices_tri_monthly.parquet")
close = pd.read_parquet(P / "prices_close_monthly.parquet")
text = pd.read_parquet(P / "text_features.parquet") if (P / "text_features.parquet").exists() else None

# drop the current, incomplete month
last_full = pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)
tri, close = tri[tri.index <= last_full], close[close.index <= last_full]
table = build_factor_table(membership[membership["month_end"] <= last_full], panel, tri, close, dcf, text)
# GICS sector of today's members (companies that left the index -> "Unknown"); used for sector-neutral ranks
wiki = pd.read_csv(settings.raw_dir / "sp500_constituents.csv")
table["sector"] = table["ticker"].map(dict(zip(wiki["ticker"].str.replace(".", "-"), wiki["sector"]))).fillna("Unknown")
table.to_parquet(P / "factor_table.parquet", index=False)
print(table.shape)
display(table.groupby(table["month_end"].dt.year)[["fwd_ret", "fscore", "earnings_yield", "dcf_mos", "mom_12_1"]]
        .apply(lambda g: g.notna().mean()).style.format("{:.0%}").set_caption("Coverage per year"))

## 1. Single factors and composite

*Composite = mean percentile rank of 6 factors. The **sector-neutral** version ranks each company only within its GICS sector, so the score picks companies, not sectors (banks always look cheap on P/E).*

In [ ]:
FACTORS = ["fscore", "gross_prof", "earnings_yield", "fcf_yield", "revenue_growth", "mom_12_1", "dcf_mos"]
results = {}
for f in FACTORS:
    table[f"{f}_signed"] = table[f] * FACTOR_DIRECTION[f]       # higher = better for every factor
    results[f] = quintile_backtest(table, f"{f}_signed", f)

COMPOSITE = ["fscore", "gross_prof", "earnings_yield", "fcf_yield", "mom_12_1", "dcf_mos"]
table["composite_raw"] = composite_score(table, COMPOSITE)
table["composite"] = composite_score(table, COMPOSITE, group_col="sector")     # sector-neutral (default)
results["composite_raw"] = quintile_backtest(table, "composite_raw", "composite (not sector-neutral)")
results["composite"] = quintile_backtest(table, "composite", "composite (sector-neutral)")

summary = pd.DataFrame([r.summary() for r in results.values()]).set_index("factor")
pct = ["Q1_ann", "Q2_ann", "Q3_ann", "Q4_ann", "Q5_ann", "LS_ann", "LS_maxdd", "Q5_vs_EW_ann", "turnover_Q5"]
display(summary.style.format({**{c: "{:.1%}" for c in pct}, "LS_sharpe": "{:.2f}", "LS_tstat": "{:.2f}",
                              "IC_mean": "{:.3f}", "IC_tstat": "{:.2f}", "avg_names": "{:.0f}"})
        .background_gradient(subset=["LS_tstat", "IC_tstat"], cmap="RdYlGn", vmin=-3, vmax=3))
quintile_bar_figure(summary).show()

spy = tri["SPY"].pct_change()
spy = spy[spy.index.isin(results["composite"].quintiles.index + pd.offsets.MonthEnd(1))]
print("SPY over the same period:", {k: round(v, 3) for k, v in perf_stats(spy).items()})

## 2. Composite score in detail

Both versions – compare the long-short lines.

In [ ]:
backtest_figure(results["composite_raw"]).show()
backtest_figure(results["composite"]).show()
ic_figure(results["composite"]).show()

## 3. Robustness

A real effect should survive (a) both halves of the sample, (b) realistic trading costs and (c) sector neutralisation.
If the raw composite works but the sector-neutral one does not, the "skill" was a **sector bet**
(e.g. overweighting cheap financials/energy), not stock picking.

In [ ]:
rows = []
mid = table["month_end"].min() + (table["month_end"].max() - table["month_end"].min()) / 2
halves = [("first half", table[table["month_end"] < mid]), ("second half", table[table["month_end"] >= mid])]
for label, part in halves:
    for col, name in [("fcf_yield_signed", "fcf_yield"), ("dcf_mos_signed", "dcf_mos"),
                      ("composite_raw", "composite (raw)"), ("composite", "composite (sector-neutral)")]:
        s = quintile_backtest(part, col, name).summary()
        rows.append({"test": label, "factor": name, "LS_ann": s["LS_ann"], "LS_tstat": s["LS_tstat"],
                     "IC_mean": s["IC_mean"], "IC_tstat": s["IC_tstat"]})
for bps in (0, 10, 25):
    for col, name in [("composite_raw", "composite (raw)"), ("composite", "composite (sector-neutral)")]:
        s = quintile_backtest(table, col, name, cost_bps=bps).summary()
        rows.append({"test": f"full period, {bps} bps costs", "factor": name, "LS_ann": s["LS_ann"],
                     "LS_tstat": s["LS_tstat"], "IC_mean": s["IC_mean"], "IC_tstat": s["IC_tstat"]})
display(pd.DataFrame(rows).style.format({"LS_ann": "{:.1%}", "LS_tstat": "{:.2f}", "IC_mean": "{:.3f}", "IC_tstat": "{:.2f}"}))

## 4. Today's ranking

The latest month's composite ranking of current members – a **starting point for research** (check them in the
explorer: fundamentals, 10-K text, interactive DCF), not a buy list.

In [ ]:
latest = table[table["month_end"] == table["month_end"].max()].dropna(subset=["composite"])
cols = ["ticker", "sector", "composite", "fscore", "earnings_yield", "fcf_yield", "dcf_mos", "mom_12_1"]
top = latest.sort_values("composite", ascending=False)[cols]
print(f"Ranking as of {latest['month_end'].max():%Y-%m-%d}, {len(latest)} companies")
display(top.head(25).style.format({"composite": "{:.2f}", "fscore": "{:.1f}", "earnings_yield": "{:.1%}",
                                   "fcf_yield": "{:.1%}", "dcf_mos": "{:+.0%}", "mom_12_1": "{:+.0%}"}))

## 5. Conclusions (results as of the first run, 2011-02 → 2026-10)

* **No single factor is significant** among S&P 500 large caps (all |t| < 2). FCF yield and DCF margin of safety point
  the right way (Q5 > Q1 by ~2.5–3 % p.a.) but are not robust across both halves.
* The **raw composite** reaches IC t-stat ≈ 2, but **disappears after sector neutralisation** and after ~10 bps costs →
  mostly a sector tilt, not stock-picking skill.
* This matches the literature: classic value/quality premia were weak in US large caps in 2011–2026 and are largely
  arbitraged away in the most-followed index. Momentum was also flat in this period.
* Practical use: the ranking is a **screen** to find candidates for deeper work (explorer → text + interactive DCF),
  not a trading strategy.

*Re-running later will change the exact numbers (new months, refreshed data).*